In [0]:
df = spark.table("workspace.default.ev_cleaned_data")

In [0]:
df

DataFrame[Station address: string, Operator: string, Total Plugs: int, Charger_Type: string, Capacity Per Plug: string, Council: string, Post Code: string, Location: struct<Latitude:string,Longitude:double>, Total Capacity(kW): int]

Dashboard


In [0]:
from pyspark.sql.functions import sum as spark_sum

total_plugs = df.agg(spark_sum("Total Plugs").alias("total_plugs"))
display(total_plugs)

total_plugs
3325


In [0]:
display(df.select("Operator").distinct().count())

30

In [0]:
display(df.select("Council").distinct().count())

124

In [0]:
display(df.count())

1108

EV Insights


In [0]:
display(df.filter(df["Charger_Type"] == "AC").count())

700

In [0]:
display(df.filter(df["Charger_Type"] == "DC").count())

408

In [0]:
display(df.count())

1108

DC Chargers are fast while AC chargers are slow in Nature

Council EV Counts

In [0]:
from pyspark.sql.functions import col, count as spark_count

min_count_df = (
    df.groupBy("Council")
      .agg(spark_count("*").alias("ev_charger_count"))
      .orderBy(col("ev_charger_count").asc())
      .limit(1)
)

display(min_count_df)

Council,ev_charger_count
Central Darling Shire Council,1


In [0]:
from pyspark.sql.functions import col, count as spark_count

max_count_df = (
    df.groupBy("Council")
      .agg(spark_count("*").alias("ev_charger_count"))
      .orderBy(col("ev_charger_count").desc())
      .limit(1)
)

display(max_count_df)

Council,ev_charger_count
"Sydney, Council of the City of",71


In [0]:
from pyspark.sql.functions import col, count as spark_count

top3_dc_councils = (
    df.filter(col("Charger_Type") == "DC")
      .groupBy("Council")
      .agg(spark_count("*").alias("dc_fast_charger_count"))
      .orderBy(col("dc_fast_charger_count").desc())
      .limit(3)
)

display(top3_dc_councils)

Council,dc_fast_charger_count
Northern Beaches Council,20
Randwick City Council,20
Blacktown City Council,16


In [0]:
top3_ac_councils = (
    df.filter(col("Charger_Type") == "AC")
      .groupBy("Council")
      .agg(spark_count("*").alias("ac_charger_count"))
      .orderBy(col("ac_charger_count").desc())
      .limit(3)
)

display(top3_ac_councils)

Council,ac_charger_count
"Sydney, Council of the City of",60
Northern Beaches Council,28
Newcastle City Council,25


In [0]:
from pyspark.sql.functions import col, sum as spark_sum

max_plugs_council = (
    df.groupBy("Council")
      .agg(spark_sum(col("Total Plugs")).alias("total_plugs"))
      .orderBy(col("total_plugs").desc())
      .limit(1)
)

display(max_plugs_council)

Council,total_plugs
"Sydney, Council of the City of",234


In [0]:
from pyspark.sql.functions import sum as spark_sum

council_total_capacity_kw = (
    df.groupBy("Council")
      .agg(spark_sum(col("Total Capacity(kW)")).alias("total_capacity_kw"))
      .orderBy(col("Council"))
)

display(council_total_capacity_kw)

council_total_capacity = (
    df.groupBy("Council")
      .agg(spark_sum(col("Total Plugs")).alias("total_capacity"))
      .orderBy(col("Council"))
)

display(council_total_capacity)

Council,total_capacity_kw
Albury City Council,3208
Armidale Regional Council,872
Ballina Shire Council,2394
Balranald Shire Council,822
Bathurst Regional Council,3414
Bayside Council,1438
Bega Valley Shire Council,370
Bellingen Shire Council,388
Berrigan Shire Council,44
Blacktown City Council,11816


Council,total_capacity
Albury City Council,32
Armidale Regional Council,11
Ballina Shire Council,29
Balranald Shire Council,7
Bathurst Regional Council,37
Bayside Council,42
Bega Valley Shire Council,15
Bellingen Shire Council,8
Berrigan Shire Council,2
Blacktown City Council,110


# Operator Market Share


In [0]:
from pyspark.sql.functions import col, count as spark_count

top3_operators = (
    df.groupBy("Operator")
      .agg(spark_count("*").alias("ev_charger_count"))
      .orderBy(col("ev_charger_count").desc())
      .limit(3)
)

display(top3_operators)

Operator,ev_charger_count
Tesla,260
Exploren,174
Chargefox,169


In [0]:
from pyspark.sql.functions import col, count as spark_count

top3_ac_operators = (
    df.filter(col("Charger_Type") == "AC")
      .groupBy("Operator")
      .agg(spark_count("*").alias("ac_charger_count"))
      .orderBy(col("ac_charger_count").desc())
      .limit(3)
)

display(top3_ac_operators)

Operator,ac_charger_count
Tesla,206
Exploren,151
Chargefox,122


In [0]:
from pyspark.sql.functions import col, sum as spark_sum

companies_most_plugs = (
    df.groupBy("Operator")
      .agg(spark_sum(col("Total Plugs")).alias("total_plugs"))
      .orderBy(col("total_plugs").desc())
)

display(companies_most_plugs)

Operator,total_plugs
Tesla,917
Chargefox,535
Exploren,434
Evie,316
NRMA,233
Non-networked,227
Ampol,121
BP,100
JOLT,95
EVX,92


In [0]:
from pyspark.sql.functions import col, sum as spark_sum

top3_companies_capacity = (
    df.groupBy("Operator")
      .agg(spark_sum(col("Total Capacity(kW)")).alias("total_capacity_kw"))
      .orderBy(col("total_capacity_kw").desc())
      .limit(3)
)

display(top3_companies_capacity)

Operator,total_capacity_kw
Tesla,90722
Evie,35300
Ampol,21100


In [0]:
from pyspark.sql.functions import col, sum as spark_sum

bottom3_operators = (
    df.groupBy("Operator")
      .agg(spark_sum(col("Total Capacity(kW)")).alias("total_capacity_kw"))
      .orderBy(col("total_capacity_kw").asc())
      .limit(3)
)

display(bottom3_operators)

Operator,total_capacity_kw
Gentari,11
EV Meter,11
BMW,12


# Charging Capacity Analysis



In [0]:
from pyspark.sql.functions import sum as spark_sum

total_capacity_kw_sum = df.agg(spark_sum(col("Total Capacity(kW)")).alias("total_capacity_kw_sum"))
display(total_capacity_kw_sum)

total_capacity_kw_sum
227553


In [0]:
from pyspark.sql.functions import col

max_plugs_station = (
    df.orderBy(col("Total Plugs").desc())
      .limit(1)
)

display(max_plugs_station)

Station address,Operator,Total Plugs,Charger_Type,Capacity Per Plug,Council,Post Code,Location,Total Capacity(kW)
2 Conferta Ave Tallawong NSW 2762,Non-networked,35,AC,22,Penrith City Council,2762,"List(-33.6923531, 150.9070683)",770


In [0]:
from pyspark.sql.functions import col

min_plugs_station = (
    df.orderBy(col("Total Plugs").asc())
      .limit(1)
)

display(min_plugs_station)

Station address,Operator,Total Plugs,Charger_Type,Capacity Per Plug,Council,Post Code,Location,Total Capacity(kW)
"1 Bradleys Head Rd, Sydney, 2088",Chargefox,1,AC,22,Mosman Municipal Council,2088,"List(-33.84103708, 151.2422452)",22


In [0]:
from pyspark.sql.functions import col

max_plugs_postcode = (
    df.groupBy("Post Code")
      .agg(spark_sum(col("Total Plugs")).alias("total_plugs"))
      .orderBy(col("total_plugs").desc())
      .limit(1)
)

display(max_plugs_postcode)

Post Code,total_plugs
2000,96


In [0]:
from pyspark.sql.functions import col

min_plugs_postcode = (
    df.orderBy(col("Total Plugs").asc())
      .select("Post Code", "Total Plugs")
      .limit(1)
)

display(min_plugs_postcode)

Post Code,Total Plugs
2088,1


In [0]:
from pyspark.sql.functions import col, sum as spark_sum

max_capacity_postcode = (
    df.groupBy("Post Code")
      .agg(spark_sum(col("Total Capacity(kW)")).alias("total_capacity_kw"))
      .orderBy(col("total_capacity_kw").desc())
      .limit(1)
)

display(max_capacity_postcode)

Post Code,total_capacity_kw
2766,8646


In [0]:
from pyspark.sql.functions import col, sum as spark_sum

min_capacity_postcode = (
    df.groupBy("Post Code")
      .agg(spark_sum(col("Total Capacity(kW)")).alias("total_capacity_kw"))
      .orderBy(col("total_capacity_kw").asc())
      .limit(1)
)

display(min_capacity_postcode)

Post Code,total_capacity_kw
2193,6
